# Exercicio 1. Análise dos primers de RT-PCR usados

## 0. Importar pacotes necessários

In [4]:
#!pip install biopython

from Bio.Blast import NCBIWWW, NCBIXML
import time
import pandas as pd

## 1. Que espécie identifica cada um dos conjuntos de primers de RT-PCR?

### 1.1. Importar dados necessários

In [2]:

# ============================================================
# Pares de primers : (name, target_organism, forward, reverse)
# ============================================================
df_primers = pd.read_csv('primer_pairs.csv', sep=';') # abre e visualiza a lista de primers usados para o RT-PCR
print('Primer List')
display(df_primers)



Primer List


,Forward,Reverse
0,AAAACGGCAAGAAAAAGCAG,ACGCGTGGTTACAGTCTTGCG
1,GTGAAATTATCGCCACGTTCGGGCAA,TCATCGCACCGTCAAAGGAACC
2,CCTTTCTAAGGAAGCGAAGGAT,AATTCTCTTCTCGGTCGCTCTA
3,GAAAGTCCAAGTTTACGCTCAAT,GCTGCACCTAAACTTACACCA
4,GCCTTCTACGTTTCCATCCA,GGCCAAATCGATTCTCAAAA
5,GCTACCACTTCAGAATCATCATC,GCACCTTCAGTCGTAGAGACG


### 1.1. Gera a sequência reversa complementar de cada primer reverso

In [ ]:
def rev_comp(seq):
    """
    Calcula o reverse complement de uma sequência de DNA.
    
    Porquê? Os primers reverse são sintetizados na direção 5'→3' 
    da cadeia complementar. Para fazer BLAST na mesma orientação 
    que o forward, precisamos do reverse complement.
    
    Exemplo: 
        Reverse original:  5'-GCACCTTCAGTCGTAGAGACG-3'
        
        Passo 1 - Complementar:  CGTGGAAGTCAGCATCTCTGC
        Passo 2 - Reverter:      CGTCTCTACGACTGAAGGTGC
        
        Reverse Complement: 5'-CGTCTCTACGACTGAAGGTGC-3'
    """
      
    # Passo 1: Criar dicionário com a conversão complementar
    comp = {'A':'T', 'T':'A', 'G':'C', 'C':'G'}
    
    # Passo 2: Complementar cada base (A↔T, G↔C)
    complementar = ""
    for base in seq:
        complementar = complementar + comp[base]
    
    print(f"Passo 2 - Complementar: 3'-{complementar}-5'")
    
    # Passo 2: Reverter a sequência (ler de trás para a frente)
    reverso = complementar[::-1]
    
    print(f"Passo 2 - Reverter:     5'-{reverso}-3'")
    
    return reverso # devolver reverse

#chamar a função rev_comp para cada primer reverse, guardando o resultado numa nova coluna do dataframe
for index, row in df_primers.iterrows():
    fwd_seq = row['Forward']
    rev_seq = row['Reverse']
    
    print(f"\n=== Analisando par: {index} ===")
    print(f"Forward: {fwd_seq}")
    print(f"Reverse: {rev_seq}")
    
    df_primers.loc[index, 'Rev_Comp'] = rev_comp(rev_seq)

#df_primers['Rev_Comp'] = df_primers['Rev_Comp'].apply(rev_comp)

display(df_primers )



=== Analisando par: 0 ===
Forward: AAAACGGCAAGAAAAAGCAG
Reverse: ACGCGTGGTTACAGTCTTGCG
Passo 2 - Complementar: 3'-TGCGCACCAATGTCAGAACGC-5'
Passo 2 - Reverter:     5'-CGCAAGACTGTAACCACGCGT-3'

=== Analisando par: 1 ===
Forward: GTGAAATTATCGCCACGTTCGGGCAA
Reverse: TCATCGCACCGTCAAAGGAACC
Passo 2 - Complementar: 3'-AGTAGCGTGGCAGTTTCCTTGG-5'
Passo 2 - Reverter:     5'-GGTTCCTTTGACGGTGCGATGA-3'

=== Analisando par: 2 ===
Forward: CCTTTCTAAGGAAGCGAAGGAT
Reverse: AATTCTCTTCTCGGTCGCTCTA
Passo 2 - Complementar: 3'-TTAAGAGAAGAGCCAGCGAGAT-5'
Passo 2 - Reverter:     5'-TAGAGCGACCGAGAAGAGAATT-3'

=== Analisando par: 3 ===
Forward: GAAAGTCCAAGTTTACGCTCAAT
Reverse: GCTGCACCTAAACTTACACCA
Passo 2 - Complementar: 3'-CGACGTGGATTTGAATGTGGT-5'
Passo 2 - Reverter:     5'-TGGTGTAAGTTTAGGTGCAGC-3'

=== Analisando par: 4 ===
Forward: GCCTTCTACGTTTCCATCCA
Reverse: GGCCAAATCGATTCTCAAAA
Passo 2 - Complementar: 3'-CCGGTTTAGCTAAGAGTTTT-5'
Passo 2 - Reverter:     5'-TTTTGAGAATCGATTTGGCC-3'

=== Analisando par: 5 ===

,Forward,Reverse,Especie_BLAST,Gene_BLAST,Rev_Comp
0,AAAACGGCAAGAAAAAGCAG,ACGCGTGGTTACAGTCTTGCG,E.coli K12,uidA,CGCAAGACTGTAACCACGCGT
1,GTGAAATTATCGCCACGTTCGGGCAA,TCATCGCACCGTCAAAGGAACC,Salmonella enterica,invA,GGTTCCTTTGACGGTGCGATGA
2,CCTTTCTAAGGAAGCGAAGGAT,AATTCTCTTCTCGGTCGCTCTA,Lactobacillus acidophilus,its,TAGAGCGACCGAGAAGAGAATT
3,GAAAGTCCAAGTTTACGCTCAAT,GCTGCACCTAAACTTACACCA,Clostridium difficile,tcdB,TGGTGTAAGTTTAGGTGCAGC
4,GCCTTCTACGTTTCCATCCA,GGCCAAATCGATTCTCAAAA,Saccharomyces cerevisiae,act1,TTTTGAGAATCGATTTGGCC
5,GCTACCACTTCAGAATCATCATC,GCACCTTCAGTCGTAGAGACG,Candida albicans,hwp1,CGTCTCTACGACTGAAGGTGC


### 1.2. Verificar a que espécie corresponde cada conjunto de primers (1 conjunto por grupo)

Instruções:
1. Vai a https://blast.ncbi.nlm.nih.gov/ → BLASTn
2. Cola o Forward do teu primer → regista a espécie e gene
    - Verifica na 'Taxonomy' qual a estirpe avaliada
    - No 'Alignments' clica no 'Graphics' para verificar que gene é sequênciada pelos primers
3. Cola o Reverse → confirma que dá hit na mesma espécie
4. Preenche abaixo usando .loc para o teu primer

In [11]:
# ============================================================
# EXERCÍCIO: Preencher após consulta no NCBI BLAST
# ============================================================

# Criar colunas vazias para os resultados
df_primers['Especie_BLAST'] = ""
df_primers['Gene_BLAST'] = ""

#E.coli K12 
df_primers.loc[0, 'Especie_BLAST'] = "E.coli K12"
df_primers.loc[0, 'Gene_BLAST'] = "uidA"
#S_enterica
df_primers.loc[1, 'Especie_BLAST'] = "Salmonella enterica"
df_primers.loc[1, 'Gene_BLAST'] = "invA"
#L_acidophilus
df_primers.loc[2, 'Especie_BLAST'] = "Lactobacillus acidophilus"
df_primers.loc[2, 'Gene_BLAST'] = "its"
#C_difficile
df_primers.loc[3, 'Especie_BLAST'] = "Clostridium difficile"
df_primers.loc[3, 'Gene_BLAST'] = "tcdB"
#S_cerevisiae
df_primers.loc[4, 'Especie_BLAST'] = "Saccharomyces cerevisiae"
df_primers.loc[4, 'Gene_BLAST'] = "act1"
#7	C_albicans
df_primers.loc[5, 'Especie_BLAST'] = "Candida albicans"
df_primers.loc[5, 'Gene_BLAST'] = "hwp1"

display(df_primers)

,Forward,Reverse,Especie_BLAST,Gene_BLAST,Rev_Comp
0,AAAACGGCAAGAAAAAGCAG,ACGCGTGGTTACAGTCTTGCG,E.coli K12,uidA,CGCAAGACTGTAACCACGCGT
1,GTGAAATTATCGCCACGTTCGGGCAA,TCATCGCACCGTCAAAGGAACC,Salmonella enterica,invA,GGTTCCTTTGACGGTGCGATGA
2,CCTTTCTAAGGAAGCGAAGGAT,AATTCTCTTCTCGGTCGCTCTA,Lactobacillus acidophilus,its,TAGAGCGACCGAGAAGAGAATT
3,GAAAGTCCAAGTTTACGCTCAAT,GCTGCACCTAAACTTACACCA,Clostridium difficile,tcdB,TGGTGTAAGTTTAGGTGCAGC
4,GCCTTCTACGTTTCCATCCA,GGCCAAATCGATTCTCAAAA,Saccharomyces cerevisiae,act1,TTTTGAGAATCGATTTGGCC
5,GCTACCACTTCAGAATCATCATC,GCACCTTCAGTCGTAGAGACG,Candida albicans,hwp1,CGTCTCTACGACTGAAGGTGC


### 1.2. Verificar a qualidade dos primers dentro dos parâmetros standerdizados

In [20]:
def analise_primer(df, index, seq, nome):
    """
    Analisa um primer de DNA:
    - Comprimento
    - Contagem de cada base
    - % GC
    - Tm (temperatura de melting) pela regra de Wallace: Tm ≈ 2(A+T) + 4(G+C)
    """
    
    # Comprimento
    comprimento = len(seq)
    df.loc[index, 'Comprimento'] = comprimento
    
    # Contar cada base
    A = seq.count('A')
    T = seq.count('T')
    G = seq.count('G')
    C = seq.count('C')
    
    # % GC
    df.loc[index, 'GC %'] = (G + C) / comprimento * 100
    
    # Tm pela regra de Wallace (válida para primers < 20 bp)
    df.loc[index, 'Tm'] = 2 * (A + T) + 4 * (G + C)
     
    return df


# ============================================================
# Analisar todos os primers usando a função analise_primer
# ============================================================
for i, row in df_primers.iterrows():
    df_primers = analise_primer(df_primers, i, row['Forward'], row['Especie_BLAST'])
    df_primers = analise_primer(df_primers, i, row['Reverse'], row['Especie_BLAST'])

display(df_primers)

,Forward,Reverse,Especie_BLAST,Gene_BLAST,Rev_Comp,Comprimento,GC %,Tm
0,AAAACGGCAAGAAAAAGCAG,ACGCGTGGTTACAGTCTTGCG,E.coli K12,uidA,CGCAAGACTGTAACCACGCGT,21.0,57.142857,66.0
1,GTGAAATTATCGCCACGTTCGGGCAA,TCATCGCACCGTCAAAGGAACC,Salmonella enterica,invA,GGTTCCTTTGACGGTGCGATGA,22.0,54.545455,68.0
2,CCTTTCTAAGGAAGCGAAGGAT,AATTCTCTTCTCGGTCGCTCTA,Lactobacillus acidophilus,its,TAGAGCGACCGAGAAGAGAATT,22.0,45.454545,64.0
3,GAAAGTCCAAGTTTACGCTCAAT,GCTGCACCTAAACTTACACCA,Clostridium difficile,tcdB,TGGTGTAAGTTTAGGTGCAGC,21.0,47.619048,62.0
4,GCCTTCTACGTTTCCATCCA,GGCCAAATCGATTCTCAAAA,Saccharomyces cerevisiae,act1,TTTTGAGAATCGATTTGGCC,20.0,40.000000,56.0
5,GCTACCACTTCAGAATCATCATC,GCACCTTCAGTCGTAGAGACG,Candida albicans,hwp1,CGTCTCTACGACTGAAGGTGC,21.0,57.142857,66.0


## 2. Qual é o grau de especificidade dos primers para o conjunto de estirpes analisadas?

In [ ]:
# ============================================================
# Organismos em análise: (nome, NCBI taxid)
# ============================================================
ORGANISMS = [
    ("E_coli_K12",    83333),
    ("S_enterica",    28901),
    ("L_acidophilus",  1579),
    ("C_difficile",    1496),
    ("S_cerevisiae",   4932),
    ("C_albicans",     5476),
]

# Parâmetros do BLAST
IDENTITY_THRESHOLD = 80.0   # % mínima de identidade para considerar um hit
EVALUE_THRESHOLD   = 10     # valor de E relaxado para sequências curtas
WORD_SIZE          = 7      # "word size" curto para primers

print(f"  {len(df_primers)} pares de primers")
print(f"  {len(ORGANISMS)} organismos")
print(f"  Total de BLASTs: {len(df_primers) * len(ORGANISMS) * 2}") # calcula o total de blasts a realizar (2 por par de primers: forward e reverse complement)

  6 pares de primers
  6 organismos
  Total de BLASTs: 72


## 3. Função BLAST

In [ ]:
def blast_primer(primer_seq, taxid, primer_name="", org_name=""):
    """
    BLAST a single primer against a specific organism.
    Returns dict with hit info.
    """
    try:
        print(f"  {primer_name} vs {org_name} (taxid:{taxid})...", end=" ", flush=True)
        
        result_handle = NCBIWWW.qblast(
            program="blastn",
            database="nt",
            sequence=primer_seq,
            entrez_query=f"txid{taxid}[ORGN]",
            word_size=WORD_SIZE,
            expect=EVALUE_THRESHOLD,
            megablast=False,
            hitlist_size=5,
        )
        
        blast_records = NCBIXML.parse(result_handle)
        blast_record = next(blast_records)
        
        best_identity = 0
        best_coverage = 0
        best_hit_name = ""
        
        for alignment in blast_record.alignments:
            for hsp in alignment.hsps:
                identity = (hsp.identities / hsp.align_length) * 100
                coverage = (hsp.align_length / len(primer_seq)) * 100
                if identity > best_identity:
                    best_identity = identity
                    best_coverage = coverage
                    best_hit_name = alignment.title[:80]
        
        has_hit = best_identity >= IDENTITY_THRESHOLD and best_coverage >= 80
        status = f"HIT! ({best_identity:.1f}%)" if has_hit else f"no ({best_identity:.1f}%)"
        print(status)
        
        return {
            "hit": has_hit,
            "identity": round(best_identity, 1),
            "coverage": round(best_coverage, 1),
            "description": best_hit_name
        }
        
    except Exception as e:
        print(f"ERROR: {e}")
        return {"hit": False, "identity": 0, "coverage": 0, "description": f"Error: {e}"}

print("BLAST function ready!")


## 4. Correr todos os BLASTs

⚠️ **Esta célula demora ~30-40 min** (96 BLASTs remotos ao NCBI).  
Se quiseres testar só alguns primers, comenta os outros no array `PRIMER_PAIRS` acima.


# Primer Specificity Checker via NCBI BLAST

Testa cada par de primers contra 6 organismos-alvo para verificar cross-reactivity.

**Organismos:**
- *E. coli* K12
- *Salmonella enterica*
- *Lactobacillus acidophilus*
- *Clostridioides difficile*
- *Saccharomyces cerevisiae*
- *Candida albicans*

**Lógica:** Para cada par, faz BLASTn do Forward e do RevComp do Reverse contra cada organismo. Se **ambos** anelam → potencial amplificação (cross-react se não for o alvo).


In [ ]:
# Store all results
all_results = []

for pair_name, target, fwd_seq, rev_seq in PRIMER_PAIRS:
    rev_comp_seq = rev_comp(rev_seq)
    
    print(f"\n{'='*60}")
    print(f"PRIMER: {pair_name}")
    print(f"  Fwd:     {fwd_seq}")
    print(f"  RevComp: {rev_comp_seq}")
    print(f"{'='*60}")
    
    for org_name, taxid in ORGANISMS:
        # BLAST Forward
        fwd_result = blast_primer(fwd_seq, taxid, f"{pair_name}_Fwd", org_name)
        time.sleep(0.5) 
        
        # BLAST RevComp of Reverse
        rev_result = blast_primer(rev_comp_seq, taxid, f"{pair_name}_RevComp", org_name)
        time.sleep(0.5)
        
        both_hit = fwd_result["hit"] and rev_result["hit"]
        is_target = (org_name == target)
        
        all_results.append({
            "Primer": pair_name,
            "Target": target,
            "Organism": org_name,
            "Is_Target": is_target,
            "Fwd_Hit": fwd_result["hit"],
            "Fwd_Identity%": fwd_result["identity"],
            "Rev_Hit": rev_result["hit"],
            "Rev_Identity%": rev_result["identity"],
            "Both_Hit": both_hit,
            "Status": "TARGET" if is_target else ("CROSS-REACT!" if both_hit else "OK")
        })

df_results = pd.DataFrame(all_results)
print("\n\nDone! Results stored in df_results")


## 5. Matriz de Especificidade

In [ ]:
def symbol(row):
    if row["Both_Hit"]:
        return "✓ BOTH"
    elif row["Fwd_Hit"]:
        return "F only"
    elif row["Rev_Hit"]:
        return "R only"
    else:
        return "·"

df_results["Symbol"] = df_results.apply(symbol, axis=1)

# Pivot to matrix
matrix = df_results.pivot_table(
    index="Primer", 
    columns="Organism", 
    values="Symbol", 
    aggfunc="first"
)

# Reorder columns
org_order = [o[0] for o in ORGANISMS]
matrix = matrix[org_order]

print("SPECIFICITY MATRIX")
print("=" * 80)
print("✓ BOTH = both primers hit (potential amplification)")
print("F only = only forward hits")
print("R only = only reverse hits") 
print("·      = no hit")
print("=" * 80)
print()
display(matrix)


## 6. Heatmap de Identidade

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for idx, direction in enumerate(["Fwd", "Rev"]):
    col = f"{direction}_Identity%"
    
    pivot = df_results.pivot_table(
        index="Primer",
        columns="Organism",
        values=col,
        aggfunc="first"
    )[org_order]
    
    ax = axes[idx]
    im = ax.imshow(pivot.values, cmap="RdYlGn_r", vmin=0, vmax=100, aspect="auto")
    
    ax.set_xticks(range(len(org_order)))
    ax.set_xticklabels(org_order, rotation=45, ha="right", fontsize=9)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index, fontsize=9)
    ax.set_title(f"{direction} Primer — % Identity", fontsize=12)
    
    # Add text annotations
    for i in range(len(pivot.index)):
        for j in range(len(org_order)):
            val = pivot.values[i, j]
            color = "white" if val > 60 else "black"
            ax.text(j, i, f"{val:.0f}", ha="center", va="center", fontsize=8, color=color)

plt.colorbar(im, ax=axes, shrink=0.8, label="% Identity")
plt.suptitle("Primer Specificity Heatmap", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("primer_specificity_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print("Heatmap saved to primer_specificity_heatmap.png")


## 7. Tabela Detalhada

In [ ]:
# Color the status column
def highlight_status(val):
    if val == "CROSS-REACT!":
        return "background-color: #ff6b6b; color: white; font-weight: bold"
    elif val == "TARGET":
        return "background-color: #51cf66; color: white; font-weight: bold"
    elif val == "OK":
        return "background-color: #f8f9fa"
    return ""

styled = df_results[["Primer", "Organism", "Fwd_Identity%", "Rev_Identity%", "Both_Hit", "Status"]].style.applymap(
    highlight_status, subset=["Status"]
)
display(styled)


## 8. Exportar Resultados

In [ ]:
# Save full results
df_results.to_csv("primer_specificity_results.csv", index=False)
print("Results saved to primer_specificity_results.csv")

# Summary: which primers are specific?
print("\n" + "="*60)
print("SUMMARY — SPECIFIC PRIMERS")
print("="*60)

for pair_name, target, fwd_seq, rev_seq in PRIMER_PAIRS:
    subset = df_results[df_results["Primer"] == pair_name]
    cross = subset[(subset["Status"] == "CROSS-REACT!")]
    target_hit = subset[(subset["Status"] == "TARGET") & (subset["Both_Hit"] == True)]
    
    if len(cross) == 0 and len(target_hit) > 0:
        print(f"  ✅ {pair_name} — SPECIFIC (hits only {target})")
    elif len(cross) > 0:
        orgs = ", ".join(cross["Organism"].tolist())
        print(f"  ❌ {pair_name} — CROSS-REACTS with: {orgs}")
    else:
        print(f"  ⚠️  {pair_name} — NO HIT on target {target}!")

print("\nDone!")
